In [1]:
import ast
import numpy as np
import pandas as pd
import torch

CSV_PATH = "data/climate_ttc/climate_2014_2023_final_with_embeddings_lag_3.csv"
MAX_ROWS = 10  # set to None or -1 to use all rows
EMBED_PREFIX = "embedding_text_lag"

# Load and parse embedding columns
df = pd.read_csv(CSV_PATH)
if MAX_ROWS is not None and MAX_ROWS >= 0:
    df = df.head(MAX_ROWS)

emb_cols = [c for c in df.columns if c.startswith(EMBED_PREFIX)]
if not emb_cols:
    raise ValueError(f"No embedding_text_lag* cols found in {CSV_PATH}")

for col in emb_cols:
    df[col] = df[col].apply(ast.literal_eval)

def concat_embeddings(row):
    return np.concatenate([np.asarray(row[c], dtype=np.float32) for c in emb_cols])

X_text = np.stack([concat_embeddings(row) for _, row in df[emb_cols].iterrows()])

# Compute attention
X = torch.tensor(X_text, dtype=torch.float32)
X_norm = torch.nn.functional.normalize(X, dim=1)
sim = X_norm @ X_norm.T
attn = torch.softmax(sim, dim=-1)

# Summary
attn_np = attn.detach().cpu().numpy()
diag = np.diag(attn_np)
off_diag = attn_np[~np.eye(attn_np.shape[0], dtype=bool)]
print(f"Attn shape: {attn_np.shape}")
print(f"Diag mean={diag.mean():.4f}, std={diag.std():.4f}")
print(f"Off-diag mean={off_diag.mean():.4f}, std={off_diag.std():.4f}")

row0 = attn_np[0]
top5 = row0.argsort()[::-1][:5]
print("Top-5 attention targets for row 0 (idx: weight):")
for idx in top5:
    print(f"  {idx}: {row0[idx]:.4f}")


Attn shape: (10, 10)
Diag mean=0.1431, std=0.0081
Off-diag mean=0.0952, std=0.0111
Top-5 attention targets for row 0 (idx: weight):
  0: 0.1650
  1: 0.1291
  2: 0.1064
  9: 0.0885
  8: 0.0866


In [2]:
df

,date,temp,precip,humidity,windspeed,text,text_lag0,text_lag1,text_lag2,text_lag3,embedding_text_lag0,embedding_text_lag1,embedding_text_lag2,embedding_text_lag3
0,2014-01-01,37.9,0.000,57.4,10.0,Arctic air is expected to invade the upper Mid...,Arctic air is expected to invade the upper Mid...,NaN,NaN,NaN,"[-0.037041038274765015, 0.06767551600933075, -...","[0.02762940712273121, -0.05966124311089516, -0...","[0.02762940712273121, -0.05966124311089516, -0...","[0.02762940712273121, -0.05966124311089516, -0..."
1,2014-01-02,35.9,0.161,82.0,11.3,"Frigid air will sweep into the Midwest, Great ...","Frigid air will sweep into the Midwest, Great ...",Arctic air is expected to invade the upper Mid...,NaN,NaN,"[-0.027351148426532745, 0.038792651146650314, ...","[-0.037041038274765015, 0.06767551600933075, -...","[0.02762940712273121, -0.05966124311089516, -0...","[0.02762940712273121, -0.05966124311089516, -0..."
2,2014-01-03,22.4,0.015,54.1,30.5,Another arctic plunge is expected for the nort...,Another arctic plunge is expected for the nort...,"Frigid air will sweep into the Midwest, Great ...",Arctic air is expected to invade the upper Mid...,NaN,"[-0.045918673276901245, 0.004100331570953131, ...","[-0.027351103723049164, 0.038793064653873444, ...","[-0.037041038274765015, 0.06767551600933075, -...","[0.02762940712273121, -0.05966124311089516, -0..."
3,2014-01-04,25.4,0.000,52.6,13.3,20°F-40°F below normal temperatures are expect...,20°F-40°F below normal temperatures are expect...,Another arctic plunge is expected for the nort...,"Frigid air will sweep into the Midwest, Great ...",Arctic air is expected to invade the upper Mid...,"[-0.050080183893442154, 0.008848625235259533, ...","[-0.045918673276901245, 0.004100331570953131, ...","[-0.027351103723049164, 0.038793064653873444, ...","[-0.037041038274765015, 0.06767551600933075, -..."
4,2014-01-05,37.1,0.041,78.7,11.0,Temperatures are expected to approach normal v...,Temperatures are expected to approach normal v...,20°F-40°F below normal temperatures are expect...,Another arctic plunge is expected for the nort...,"Frigid air will sweep into the Midwest, Great ...","[-0.010337048210203648, 0.0067078773863613605,...","[-0.050080183893442154, 0.008848625235259533, ...","[-0.045918673276901245, 0.004100331570953131, ...","[-0.027351148426532745, 0.038792651146650314, ..."
5,2014-01-06,36.6,0.107,63.7,24.9,Overview of upper pattern transition from zona...,Overview of upper pattern transition from zona...,Temperatures are expected to approach normal v...,20°F-40°F below normal temperatures are expect...,Another arctic plunge is expected for the nort...,"[-0.009884157218039036, 0.04276413843035698, -...","[-0.010337048210203648, 0.0067078773863613605,...","[-0.050080183893442154, 0.008848625235259533, ...","[-0.04591866210103035, 0.004100150428712368, -..."
6,2014-01-07,12.9,0.005,33.3,18.5,The medium range forecast shows a rebuilding r...,The medium range forecast shows a rebuilding r...,Overview of upper pattern transition from zona...,Temperatures are expected to approach normal v...,20°F-40°F below normal temperatures are expect...,"[0.02535320073366165, 0.06275518238544464, -0....","[-0.009884157218039036, 0.04276413843035698, -...","[-0.010337048210203648, 0.0067078773863613605,...","[-0.050080183893442154, 0.008848625235259533, ..."
7,2014-01-08,22.2,0.000,40.4,7.8,Above normal temperatures are expected across ...,Above normal temperatures are expected across ...,The medium range forecast shows a rebuilding r...,Overview of upper pattern transition from zona...,Temperatures are expected to approach normal v...,"[-0.0052687847055494785, 0.04374668747186661, ...","[0.02535320073366165, 0.06275518238544464, -0....","[-0.009884157218039036, 0.04276413843035698, -...","[-0.010337048210203648, 0.0067078773863613605,..."
8,2014-01-09,33.2,0.000,55.8,8.1,Transition to a ridge/trough/ridge flow patter...,Transition to a ridge/trough/ridge flow patter...,Above normal temperatures are expect

In [4]:
"""
Quick check: load the climate lagged-embedding dataset and compute the external text
attention matrix (cosine-softmax) used by NanoTabPFNRegressor._compute_attn_weight_external.
"""

from __future__ import annotations

import argparse
import ast
from typing import Sequence

import numpy as np
import pandas as pd
import torch

DATA_PATH = "data/climate_ttc/climate_2014_2023_final_with_embeddings_lag_3.csv"
EMBED_PREFIX = "embedding_text_lag"


def load_embeddings(path: str, max_rows: int | None = None) -> np.ndarray:
    """Load and concatenate all embedding_text_lag* columns into one array per row."""
    df = pd.read_csv(path)
    if max_rows is not None:
        df = df.head(max_rows)

    emb_cols = [c for c in df.columns if c.startswith(EMBED_PREFIX)]
    if not emb_cols:
        raise ValueError(f"No columns starting with '{EMBED_PREFIX}' found in {path}")

    # Parse list-like strings back to float arrays
    for col in emb_cols:
        df[col] = df[col].apply(ast.literal_eval)

    def concat_embeddings(row: pd.Series) -> np.ndarray:
        return np.concatenate([np.asarray(row[c], dtype=np.float32) for c in emb_cols])

    X_text = np.stack([concat_embeddings(row) for _, row in df[emb_cols].iterrows()])
    print(f"Loaded {len(df)} rows with embedding cols {emb_cols}")
    print(f"Per-lag embedding dim: {len(df[emb_cols[0]].iloc[0])}, concatenated dim: {X_text.shape[1]}")
    return X_text


def compute_text_attention(embeddings: np.ndarray) -> torch.Tensor:
    """Cosine similarity + softmax over rows, without self-attention on the diagonal."""
    X = torch.tensor(embeddings, dtype=torch.float32)
    X_norm = torch.nn.functional.normalize(X, dim=1)
    sim = X_norm @ X_norm.T
    if sim.shape[0] > 1:
        sim = sim.masked_fill(torch.eye(sim.shape[0], device=sim.device, dtype=torch.bool), float("-inf"))
    attn = torch.softmax(sim, dim=-1)
    return attn


def summarize_attention(attn: torch.Tensor) -> None:
    """Print basic stats and a sample row's top weights."""
    attn_np = attn.detach().cpu().numpy()
    diag = np.diag(attn_np)
    off_diag = attn_np[~np.eye(attn_np.shape[0], dtype=bool)]

    print(f"Attn shape: {attn_np.shape}")
    print(f"Diagonal weight mean={diag.mean():.4f}, std={diag.std():.4f}")
    print(f"Off-diagonal weight mean={off_diag.mean():.4f}, std={off_diag.std():.4f}")

    row0 = attn_np[0]
    top5_idx = row0.argsort()[::-1][:5]
    print("Top-5 attention targets for row 0 (idx: weight):")
    for idx in top5_idx:
        print(f"  {idx}: {row0[idx]:.4f}")


def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--path",
        default=DATA_PATH,
        help=f"Path to CSV with embeddings (default: {DATA_PATH})",
    )
    parser.add_argument(
        "--max-rows",
        type=int,
        default=128,
        help="Limit rows for a quick test (default: 128, use -1 for all rows).",
    )
    return parser.parse_args(argv)


def main(argv: Sequence[str] | None = None) -> None:
    args = parse_args(argv)
    max_rows = None if args.max_rows is None or args.max_rows < 0 else args.max_rows

    embeddings = load_embeddings(args.path, max_rows=max_rows)
    attn = compute_text_attention(embeddings)
    summarize_attention(attn)



In [ ]:
embeddings = load_embeddings(DATA_PATH, max_rows=10)
attn = compute_text_attention(embeddings)
summarize_attention(attn)

Loaded 10 rows with embedding cols ['embedding_text_lag0', 'embedding_text_lag1', 'embedding_text_lag2', 'embedding_text_lag3']
Per-lag embedding dim: 1024, concatenated dim: 4096
Attn shape: (10, 10)
Diagonal weight mean=0.0000, std=0.0000
Off-diagonal weight mean=0.1111, std=0.0129
Top-5 attention targets for row 0 (idx: weight):
  1: 0.1546
  2: 0.1274
  9: 0.1059
  8: 0.1038
  3: 0.1031


In [8]:
embeddings[0]

array([-0.03704104,  0.06767552, -0.00640702, ..., -0.06938682,
       -0.10676806,  0.00248934], shape=(4096,), dtype=float32)

In [11]:
print(attn[0].shape)
attn[0]

torch.Size([10])


tensor([0.0000, 0.1546, 0.1274, 0.1031, 0.1029, 0.0982, 0.1015, 0.1025, 0.1038,
        0.1059])

### Align dimension between attn_weight and text_attn

In [12]:
import torch

def compute_text_attn(X_text: torch.Tensor) -> torch.Tensor:
    """
    X_text: [B, L, F_text] text embeddings (train rows first, then test rows).
    Returns: attn_weight_external [B, L, L] (softmax over keys).
    """
    X_norm = torch.nn.functional.normalize(X_text, dim=-1)
    sim = torch.einsum("bld,bmd->blm", X_norm, X_norm)  # [B, L, L]
    return torch.softmax(sim, dim=-1)

# Example
B, L, F_text = 1, 5, 8
X_text = torch.randn(B, L, F_text)
attn_weight_external = compute_text_attn(X_text)
print(attn_weight_external.shape)  # torch.Size([1, 5, 5])


torch.Size([1, 5, 5])


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm
import sklearn
# from tabpfn import TabPFNRegressor
# from tabpfn.finetune_utils import clone_model_for_evaluation
# from tabpfn.utils import meta_dataset_collator
from functools import partial


def prepare_data(config: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Loads, subsets, and splits the California Housing dataset."""
    print("--- 1. Data Preparation ---")
    # Fetch Ames housing data from OpenML
    bike_sharing = sklearn.datasets.fetch_openml(
        name="Bike_Sharing_Demand", version=2, as_frame=True, parser="auto"
    )

    # Separate features (X) and target (y)
    X_df = bike_sharing.data
    y_df = bike_sharing.target

    # Select only numeric features for simplicity
    X_numeric = X_df.select_dtypes(include=np.number)

    X_all, y_all = X_numeric.values, y_df.values

    rng = np.random.default_rng(config["random_seed"])
    num_samples_to_use = min(config["num_samples_to_use"], len(y_all))
    indices = rng.choice(np.arange(len(y_all)), size=num_samples_to_use, replace=False)
    X, y = X_all[indices], y_all[indices]

    splitter = partial(
        train_test_split,
        test_size=config["valid_set_ratio"],
        random_state=config["random_seed"],
    )
    X_train, X_test, y_train, y_test = splitter(X, y)

    print(
        f"Loaded and split data: {X_train.shape[0]} train, {X_test.shape[0]} test samples."
    )
    print("---------------------------\n")
    return X_train, X_test, y_train, y_test

In [18]:
prepare_data({ "random_seed": 42, "num_samples_to_use": 1000, "valid_set_ratio": 0.2 })

--- 1. Data Preparation ---
Loaded and split data: 800 train, 200 test samples.
---------------------------



(array([[ 0.    ,  7.    ,  2.    , ..., 33.335 ,  0.54  ,  7.0015],
        [ 0.    ,  1.    , 19.    , ..., 12.88  ,  0.38  ,  7.0015],
        [ 1.    ,  1.    ,  2.    , ..., 21.21  ,  0.94  ,  8.9981],
        ...,
        [ 0.    , 11.    ,  8.    , ..., 25.76  ,  0.68  ,  7.0015],
        [ 1.    ,  9.    ,  3.    , ..., 21.97  ,  0.77  ,  0.    ],
        [ 1.    , 12.    ,  4.    , ..., 20.455 ,  0.94  ,  7.0015]],
       shape=(800, 8)),
 array([[ 1.    ,  4.    , 11.    , ..., 31.06  ,  0.33  , 27.9993],
        [ 0.    ,  8.    , 20.    , ..., 34.85  ,  0.79  , 19.0012],
        [ 1.    ,  2.    ,  6.    , ..., 15.15  ,  0.75  ,  6.0032],
        ...,
        [ 0.    , 11.    , 15.    , ..., 25.76  ,  0.6   ,  8.9981],
        [ 0.    ,  2.    , 16.    , ..., 18.18  ,  0.29  ,  6.0032],
        [ 1.    ,  2.    , 10.    , ..., 21.21  ,  0.77  ,  7.0015]],
       shape=(200, 8)),
 array([  7, 132,  13,   1,   4,   8,   9, 123, 428, 372, 169, 704,   5,
        112, 162, 109, 

In [19]:
# Notebook snippet: test NanoTabPFNRegressor._compute_text_similarity_external and _compute_attn_weight_external
# This avoids needing pfns/h5py by stubbing minimal modules and loading tfmplayground/interface.py directly.

import importlib.util
import sys
import types
from pathlib import Path

import numpy as np
import torch


def ensure_pfns_stub():
    if "pfns" in sys.modules:
        return
    pfns_mod = types.ModuleType("pfns")
    bar_mod = types.ModuleType("pfns.bar_distribution")

    class FullSupportBarDistribution:
        def __init__(self, *args, **kwargs):
            raise RuntimeError("Stub. Install `pfns` to use full regressor.")

    bar_mod.FullSupportBarDistribution = FullSupportBarDistribution
    sys.modules["pfns"] = pfns_mod
    sys.modules["pfns.bar_distribution"] = bar_mod


def load_interface_module(repo_root: Path):
    # Stub package so absolute imports inside interface.py work.
    pkg = types.ModuleType("tfmplayground")
    pkg.__path__ = [str(repo_root / "tfmplayground")]
    sys.modules.setdefault("tfmplayground", pkg)

    # Stub utils to avoid importing h5py/pfns helpers.
    utils_mod = types.ModuleType("tfmplayground.utils")

    def get_default_device():
        return "cpu"

    utils_mod.get_default_device = get_default_device
    sys.modules.setdefault("tfmplayground.utils", utils_mod)

    interface_path = repo_root / "tfmplayground" / "interface.py"
    spec = importlib.util.spec_from_file_location("tfmplayground.interface", interface_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules["tfmplayground.interface"] = module
    spec.loader.exec_module(module)
    return module


# --- run the test ---
ensure_pfns_stub()
repo_root = Path.cwd()  # assumes notebook is run from repo root
interface = load_interface_module(repo_root)
NanoTabPFNRegressor = interface.NanoTabPFNRegressor

# Create an instance without running __init__ (we only need device + train_text)
reg = NanoTabPFNRegressor.__new__(NanoTabPFNRegressor)
reg.device = torch.device("cpu")

# train_text / text_test are (N, L, D) = (num_rows, num_text_features, embedding_size)
n_train, n_test, L, D = 5, 2, 4, 8
rng = np.random.default_rng(0)
train_text = rng.standard_normal(size=(n_train, L, D), dtype=np.float32)
text_test = rng.standard_normal(size=(n_test, L, D), dtype=np.float32)

# Make test row 0 identical to train row 0 -> cosine similarity should be ~1 for all L features
text_test[0] = train_text[0]
reg.train_text = train_text

sim = reg._compute_text_similarity_external(text_test)  # (L, N_test, N_train)
attn = reg._compute_attn_weight_external(text_test)     # (1, N_test, N_train)

print("sim shape:", tuple(sim.shape))       # expected (L, N_test, N_train)
print("attn shape:", tuple(attn.shape))     # expected (1, N_test, N_train)
print("attn row sums:", attn.sum(dim=-1))   # should be ~1.0

print("cosine(sim) for test[0] vs train[0] across L features:",
      sim[:, 0, 0].detach().cpu().numpy())

print("attn weights for test[0] over train rows:",
      np.round(attn[0, 0].detach().cpu().numpy(), 4))
print("argmax train idx for test[0]:", int(attn[0, 0].argmax()))


sim shape: (4, 2, 5)
attn shape: (1, 2, 5)
attn row sums: tensor([[1.0000, 1.0000]])
cosine(sim) for test[0] vs train[0] across L features: [1.0000001 1.        0.9999999 0.9999999]
attn weights for test[0] over train rows: [0.4347 0.1555 0.1516 0.1502 0.1081]
argmax train idx for test[0]: 0


In [24]:
sim[:, 1, 0]

tensor([-9.9522e-05, -4.9628e-02,  2.2806e-01,  2.8024e-01])

In [21]:
attn

tensor([[[0.4347, 0.1555, 0.1516, 0.1502, 0.1081],
         [0.2228, 0.1931, 0.2446, 0.1734, 0.1661]]])